In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark import SparkConf

spark_jars = os.environ.get("SPARK_JARS", "")
jar_list = spark_jars.split(",") if spark_jars else []
s3a_endpoint = os.environ.get("S3A_ENDPOINT", "")
s3a_access_key = os.environ.get("S3A_ACCESS_KEY", "")
s3a_secret_key = os.environ.get("S3A_SECRET_KEY", "")
driver_memory = os.environ.get("SPARK_DRIVER_MEMORY", "4g")

spark = (
    SparkSession.builder
    .appName("Tazama-Dashboard")
    .master("local[*]")
    .config("spark.jars", spark_jars)
    .config("spark.driver.extraClassPath", ":".join(jar_list))
    .config("spark.executor.extraClassPath", ":".join(jar_list))
    .config("spark.hadoop.fs.s3a.endpoint", s3a_endpoint)
    .config("spark.hadoop.fs.s3a.access.key", s3a_access_key)
    .config("spark.hadoop.fs.s3a.secret.key", s3a_secret_key)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.impl.disable.cache", "true")
    .config("spark.hadoop.fs.s3a.connection.maximum", "100")
    .config("spark.hadoop.fs.s3a.fast.upload", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryo.registrator", "org.apache.spark.HoodieSparkKryoRegistrar")
    .config("spark.sql.extensions", "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog")
    .config("spark.driver.memory", driver_memory)
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.default.parallelism", "16")
    .config("spark.memory.fraction", "0.8")
    .config("spark.memory.storageFraction", "0.2")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128mb")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark Version: {spark.version}")

Spark Version: 3.4.2


In [13]:
WAREHOUSE_ROOT = os.environ.get("WAREHOUSE_ROOT", "/opt/Tazama_Warehouse")

alerts_df  = spark.read.format("hudi").load(f"{WAREHOUSE_ROOT}/gold/alerts")
cases_df   = spark.read.format("hudi").load(f"{WAREHOUSE_ROOT}/gold/cases")
tasks_df   = spark.read.format("hudi").load(f"{WAREHOUSE_ROOT}/gold/tasks")
transactions_df = spark.read.format("hudi").load(f"{WAREHOUSE_ROOT}/gold/transactions")

In [14]:
import pyspark.sql.functions as F
from pyspark.sql import Window
from datetime import date, timedelta

def resolve_date_range(period: str = None, start_date: str = None, end_date: str = None):
    """
    Returns (start_date, end_date) as strings YYYY-MM-DD. end_date inclusive.
    """
    today = date.today()

    if start_date and end_date:
        return start_date, end_date

    period = (period or "LAST_30_DAYS").upper()

    if period == "LAST_7_DAYS":
        return str(today - timedelta(days=6)), str(today)
    if period == "LAST_30_DAYS":
        return str(today - timedelta(days=29)), str(today)
    if period == "LAST_90_DAYS":
        return str(today - timedelta(days=89)), str(today)
    if period in ("LAST_YEAR", "YEAR"):
        return str(today - timedelta(days=364)), str(today)
    if period == "ALL_HISTORY":
        return None, None

    raise ValueError(f"Unknown period: {period}")

def apply_date_filter(df, ts_col: str, start_date: str, end_date: str):
    """
    Inclusive date filter on timestamp/date column.
    """
    if start_date is None and end_date is None:
        return df
    d = df.withColumn("_d", F.to_date(F.col(ts_col)))
    return d.filter((F.col("_d") >= F.lit(start_date)) & (F.col("_d") <= F.lit(end_date))).drop("_d")

def add_time_bucket(df, ts_col: str, granularity: str):
    """
    Adds 'period' column based on granularity: daily/weekly/monthly/quarterly
    """
    g = granularity.lower()
    ts = F.col(ts_col)

    if g == "daily":
        period = F.to_date(ts)
    elif g == "weekly":
        period = F.date_trunc("week", ts)
    elif g == "monthly":
        period = F.date_trunc("month", ts)
    elif g == "quarterly":
        # quarter start: truncate to quarter via year + quarter math
        period = F.expr("make_date(year({0}), ((quarter({0})-1)*3)+1, 1)".format(ts_col))
    else:
        raise ValueError("granularity must be one of: daily/weekly/monthly/quarterly")

    return df.withColumn("period", period)


In [15]:
def build_fraud_trend_timeseries(
    alerts,
    cases,
    period="LAST_90_DAYS",
    granularity="weekly",
    start_date=None,
    end_date=None,
    # Interdiction / blocking mapping (default derived from your sample)
    interdiction_predicate=None,
    blocking_predicate=None
):
    """
    Returns Spark DataFrame with one row per time bucket ('period') containing:
    - Alert trend metrics
    - Detection effectiveness metrics
    - Fraud value metrics
    - Also includes raw counts to support drilldowns later
    """

    start_date, end_date = resolve_date_range(period, start_date, end_date)

    # ----------------------------
    # Defaults based on your Gold schema + sample
    # ----------------------------
    # "Interdiction" / "blocking" is not explicit in columns you showed.
    # Best available proxy: priority_norm == 'BREACH'
    if interdiction_predicate is None:
        interdiction_predicate = lambda a: (F.col("alert_status") == "ALRT") & (F.col("priority_norm") == "BREACH")
    if blocking_predicate is None:
        blocking_predicate = lambda a: (F.col("alert_status") == "ALRT") & (F.col("priority_norm") == "BREACH")

    # ----------------------------
    # ALERTS base filtered + bucketed
    # Use created_at_ts as "evaluation output time" since it's when evaluation result exists
    # ----------------------------
    a = apply_date_filter(alerts, "created_at_ts", start_date, end_date)
    a = a.withColumn("alert_type_u", F.upper(F.col("alert_type_norm")))
    a = with_period = add_time_bucket(a, "created_at_ts", granularity)

    # Treat each row as a transaction evaluation result (ALRT or NALT)
    # evaluated_tx = count of rows
    evaluated = (
        a.groupBy("period")
         .agg(
             F.count("*").alias("tx_evaluated"),

             # alert fired
             F.sum(F.when(F.col("alert_status") == "ALRT", 1).otherwise(0)).alias("tx_alerted"),

             # AML / Fraud breakdown by alert_type_norm (only meaningful for ALRT)
             F.sum(F.when((F.col("alert_status") == "ALRT") & (F.col("alert_type_u") == "AML"), 1).otherwise(0)).alias("aml_alerts"),
             F.sum(F.when((F.col("alert_status") == "ALRT") & (F.col("alert_type_u") == "FRAUD"), 1).otherwise(0)).alias("fraud_alerts"),
             F.sum(F.when((F.col("alert_status") == "ALRT") & (F.col("alert_type_u") == "FRAUD_AND_AML"), 1).otherwise(0)).alias("fraud_aml_alerts"),

             # Interdiction / blocking (proxy by BREACH unless you override)
             F.sum(F.when(interdiction_predicate(a), 1).otherwise(0)).alias("interdiction_alerts"),
             F.sum(F.when(blocking_predicate(a), 1).otherwise(0)).alias("blocking_alerts"),

             # Outcomes for effectiveness metrics
             F.sum(F.when(F.col("prediction_outcome_norm") == "TRUE_POSITIVE", 1).otherwise(0)).alias("tp"),
             F.sum(F.when(F.col("prediction_outcome_norm") == "FALSE_POSITIVE", 1).otherwise(0)).alias("fp"),
             F.sum(F.when(F.col("prediction_outcome_norm") == "TRUE_NEGATIVE", 1).otherwise(0)).alias("tn"),
             F.sum(F.when(F.col("prediction_outcome_norm") == "FALSE_NEGATIVE", 1).otherwise(0)).alias("fn"),

             # Timing: evaluation processing time proxies
             F.avg(F.col("event_to_ingest_ms").cast("double")).alias("avg_event_to_ingest_ms"),
             F.avg(F.col("total_processing_time_ms").cast("double")).alias("avg_total_processing_ms"),

             # Lag: event_ts → created_at_ts (detection system latency)
             F.avg((F.unix_timestamp("created_at_ts") - F.unix_timestamp("event_ts"))/3600.0).alias("avg_eval_lag_hours"),

             # Fraud value proxies (if tx_amount is populated)
             F.sum(F.when(
                 (F.col("prediction_outcome_norm").isin("TRUE_POSITIVE", "FALSE_NEGATIVE")),
                 F.col("tx_amount").cast("double")
             ).otherwise(F.lit(0.0))).alias("total_fraud_value"),

             F.sum(F.when(
                 (F.col("prediction_outcome_norm") == "TRUE_POSITIVE") & interdiction_predicate(a),
                 F.col("tx_amount").cast("double")
             ).otherwise(F.lit(0.0))).alias("prevented_fraud_value")
         )
    )

    # ----------------------------
    # DERIVE rates safely
    # ----------------------------
    e = evaluated

    e = (e
        .withColumn("evaluated_alert_rate", F.when(F.col("tx_evaluated") > 0, F.col("tx_alerted") / F.col("tx_evaluated")).otherwise(F.lit(0.0)))
        .withColumn("aml_alert_rate", F.when(F.col("tx_evaluated") > 0, F.col("aml_alerts") / F.col("tx_evaluated")).otherwise(F.lit(0.0)))
        .withColumn("fraud_alert_rate", F.when(F.col("tx_evaluated") > 0, F.col("fraud_alerts") / F.col("tx_evaluated")).otherwise(F.lit(0.0)))
        .withColumn("interdiction_rate", F.when(F.col("tx_evaluated") > 0, F.col("interdiction_alerts") / F.col("tx_evaluated")).otherwise(F.lit(0.0)))
        .withColumn("block_rate", F.when(F.col("tx_alerted") > 0, F.col("blocking_alerts") / F.col("tx_alerted")).otherwise(F.lit(0.0)))

        # Effectiveness
        .withColumn("fraud_rate", F.when(F.col("tx_evaluated") > 0, (F.col("tp")+F.col("fn")) / F.col("tx_evaluated")).otherwise(F.lit(0.0)))
        .withColumn("precision", F.when((F.col("tp")+F.col("fp")) > 0, F.col("tp") / (F.col("tp")+F.col("fp"))).otherwise(F.lit(0.0)))
        .withColumn("tpr_recall", F.when((F.col("tp")+F.col("fn")) > 0, F.col("tp") / (F.col("tp")+F.col("fn"))).otherwise(F.lit(0.0)))
        .withColumn("fpr", F.when((F.col("fp")+F.col("tn")) > 0, F.col("fp") / (F.col("fp")+F.col("tn"))).otherwise(F.lit(0.0)))
        .withColumn("fn_rate", F.when(F.col("tx_evaluated") > 0, F.col("fn") / F.col("tx_evaluated")).otherwise(F.lit(0.0)))
        .withColumn("manual_detection_rate", F.when((F.col("tp")+F.col("fn")) > 0, F.col("fn") / (F.col("tp")+F.col("fn"))).otherwise(F.lit(0.0)))
    )

    e = e.withColumn(
        "f1",
        F.when((F.col("precision")+F.col("tpr_recall")) > 0,
               (2 * F.col("precision") * F.col("tpr_recall")) / (F.col("precision")+F.col("tpr_recall"))
        ).otherwise(F.lit(0.0))
    )

    # Avg fraud value = total fraud value / count of confirmed fraud transactions (tp+fn)
    e = e.withColumn(
        "avg_fraud_value",
        F.when((F.col("tp")+F.col("fn")) > 0, F.col("total_fraud_value") / (F.col("tp")+F.col("fn"))).otherwise(F.lit(0.0))
    )

    # ----------------------------
    # MTTD from CASES (closed with disposition)
    # ----------------------------
    CLOSED_WITH_DISPOSITION = [
        "STATUS_81_CLOSED_REFUTED",
        "STATUS_82_CLOSED_CONFIRMED",
        "STATUS_83_CLOSED_INCONCLUSIVE",
        "STATUS_71_AUTOCLOSED_CONFIRMED",
        "STATUS_72_AUTOCLOSED_REFUTED",
    ]

    c = cases.filter(F.col("status").isin(CLOSED_WITH_DISPOSITION))
    c = apply_date_filter(c, "case_updated_date", start_date, end_date)
    c = add_time_bucket(c, "case_updated_ts", granularity)

    mttd = (
        c.groupBy("period")
         .agg(F.avg((F.col("created_to_updated_ms")/ (1000*60*60*24)).cast("double")).alias("mttd_days"))
    )

    # Join MTTD in
    out = (
        e.join(mttd, "period", "left")
         .fillna({"mttd_days": 0.0})
         .orderBy("period")
    )

    return out


In [16]:
def add_pop_indicators(trend_df):
    w = Window.orderBy("period")

    df = (trend_df
        .withColumn("prev_aml_alert_rate", F.lag("aml_alert_rate").over(w))
        .withColumn("prev_fraud_alert_rate", F.lag("fraud_alert_rate").over(w))
        .withColumn("prev_precision", F.lag("precision").over(w))
        .withColumn("prev_tpr", F.lag("tpr_recall").over(w))
        .withColumn("prev_fraud_rate", F.lag("fraud_rate").over(w))
        .withColumn("prev_total_fraud_value", F.lag("total_fraud_value").over(w))
    )

    # velocity (%) = (current-prev)/prev
    df = (df
        .withColumn("aml_alert_velocity_pct",
            F.when(F.col("prev_aml_alert_rate") > 0,
                   (F.col("aml_alert_rate") - F.col("prev_aml_alert_rate")) / F.col("prev_aml_alert_rate") * 100.0
            ).otherwise(F.lit(0.0))
        )
        .withColumn("fraud_alert_velocity_pct",
            F.when(F.col("prev_fraud_alert_rate") > 0,
                   (F.col("fraud_alert_rate") - F.col("prev_fraud_alert_rate")) / F.col("prev_fraud_alert_rate") * 100.0
            ).otherwise(F.lit(0.0))
        )
    )

    return df


In [17]:
def add_pop_indicators(trend_df):
    w = Window.orderBy("period")

    df = (trend_df
        .withColumn("prev_aml_alert_rate", F.lag("aml_alert_rate").over(w))
        .withColumn("prev_fraud_alert_rate", F.lag("fraud_alert_rate").over(w))
        .withColumn("prev_precision", F.lag("precision").over(w))
        .withColumn("prev_tpr", F.lag("tpr_recall").over(w))
        .withColumn("prev_fraud_rate", F.lag("fraud_rate").over(w))
        .withColumn("prev_total_fraud_value", F.lag("total_fraud_value").over(w))
    )

    # velocity (%) = (current-prev)/prev
    df = (df
        .withColumn("aml_alert_velocity_pct",
            F.when(F.col("prev_aml_alert_rate") > 0,
                   (F.col("aml_alert_rate") - F.col("prev_aml_alert_rate")) / F.col("prev_aml_alert_rate") * 100.0
            ).otherwise(F.lit(0.0))
        )
        .withColumn("fraud_alert_velocity_pct",
            F.when(F.col("prev_fraud_alert_rate") > 0,
                   (F.col("fraud_alert_rate") - F.col("prev_fraud_alert_rate")) / F.col("prev_fraud_alert_rate") * 100.0
            ).otherwise(F.lit(0.0))
        )
    )

    return df


In [18]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def _trend_arrow(curr, prev, higher_is_better=True):
    if prev is None or prev == 0:
        return "—"
    if curr == prev:
        return "→"
    up = curr > prev
    if higher_is_better:
        return "↑" if up else "↓"
    else:
        return "↓" if up else "↑"

def render_fraud_trend_dashboard(trend_sdf, title="Fraud Trend Analysis Dashboard"):
    # Convert to pandas safely (cast period to string to avoid datetime64 precision issues)
    trend_pd = trend_sdf.withColumn("period_str", F.col("period").cast("string")).drop("period").toPandas()
    if len(trend_pd) == 0:
        print("No data for selected range.")
        return

    trend_pd = trend_pd.sort_values("period_str")

    # latest + previous
    latest = trend_pd.iloc[-1]
    prev = trend_pd.iloc[-2] if len(trend_pd) >= 2 else None

    # ---- KPI cards (current value + arrow) ----
    kpis = [
        ("Evaluated Alert Rate", latest["evaluated_alert_rate"]*100, "%", True, prev["evaluated_alert_rate"]*100 if prev is not None else None),
        ("AML Alert Rate", latest["aml_alert_rate"]*100, "%", True, prev["aml_alert_rate"]*100 if prev is not None else None),
        ("Fraud Alert Rate", latest["fraud_alert_rate"]*100, "%", True, prev["fraud_alert_rate"]*100 if prev is not None else None),
        ("Interdiction Rate", latest["interdiction_rate"]*100, "%", True, prev["interdiction_rate"]*100 if prev is not None else None),
        ("Block Rate", latest["block_rate"]*100, "%", False, prev["block_rate"]*100 if prev is not None else None),

        ("Fraud Rate", latest["fraud_rate"]*100, "%", False, prev["fraud_rate"]*100 if prev is not None else None),
        ("Precision", latest["precision"]*100, "%", True, prev["precision"]*100 if prev is not None else None),
        ("TPR / Recall", latest["tpr_recall"]*100, "%", True, prev["tpr_recall"]*100 if prev is not None else None),
        ("F1", latest["f1"]*100, "%", True, prev["f1"]*100 if prev is not None else None),
        ("Missed Fraud Rate", latest["fn_rate"]*100, "%", False, prev["fn_rate"]*100 if prev is not None else None),

        ("Total Fraud Value", latest["total_fraud_value"], "", False, prev["total_fraud_value"] if prev is not None else None),
        ("Prevented Fraud Value", latest["prevented_fraud_value"], "", True, prev["prevented_fraud_value"] if prev is not None else None),
    ]

    fig_kpis = make_subplots(rows=4, cols=3, specs=[[{"type":"indicator"}]*3]*4,
                            vertical_spacing=0.12, horizontal_spacing=0.12)

    for i, (name, val, unit, hib, prev_val) in enumerate(kpis):
        r = (i // 3) + 1
        c = (i % 3) + 1
        arrow = _trend_arrow(val, prev_val, higher_is_better=hib)

        fig_kpis.add_trace(
            go.Indicator(
                mode="number",
                value=float(val) if val is not None else 0.0,
                title={"text": f"{name} <br><sub>{arrow}</sub>"},
                number={"suffix": unit}
            ),
            row=r, col=c
        )

    fig_kpis.update_layout(
        title_text=f"<b>{title}</b><br><sub>Latest Period: {latest['period_str']}</sub>",
        height=1100, width=1400, showlegend=False
    )

    fig_kpis.show()

    # ---- Time-series charts (3 panes like your Example 2) ----
    fig_trends = make_subplots(
        rows=3, cols=1,
        subplot_titles=(
            "Alert Trend Metrics",
            "Detection Effectiveness Metrics",
            "Fraud Value Metrics"
        ),
        vertical_spacing=0.12,
        specs=[[{"type":"xy"}],[{"type":"xy"}],[{"type":"xy"}]]
    )

    # Row 1: Alert trends
    fig_trends.add_trace(go.Scatter(x=trend_pd["period_str"], y=trend_pd["evaluated_alert_rate"]*100,
                                    mode="lines+markers", name="Evaluated Alert Rate (%)"), row=1, col=1)
    fig_trends.add_trace(go.Scatter(x=trend_pd["period_str"], y=trend_pd["aml_alert_rate"]*100,
                                    mode="lines+markers", name="AML Alert Rate (%)"), row=1, col=1)
    fig_trends.add_trace(go.Scatter(x=trend_pd["period_str"], y=trend_pd["fraud_alert_rate"]*100,
                                    mode="lines+markers", name="Fraud Alert Rate (%)"), row=1, col=1)
    fig_trends.add_trace(go.Scatter(x=trend_pd["period_str"], y=trend_pd["interdiction_rate"]*100,
                                    mode="lines+markers", name="Interdiction Rate (%)"), row=1, col=1)
    fig_trends.add_trace(go.Scatter(x=trend_pd["period_str"], y=trend_pd["block_rate"]*100,
                                    mode="lines+markers", name="Block Rate (%)"), row=1, col=1)

    # Row 2: Effectiveness
    fig_trends.add_trace(go.Scatter(x=trend_pd["period_str"], y=trend_pd["fraud_rate"]*100,
                                    mode="lines+markers", name="Fraud Rate (%)"), row=2, col=1)
    fig_trends.add_trace(go.Scatter(x=trend_pd["period_str"], y=trend_pd["precision"]*100,
                                    mode="lines+markers", name="Precision (%)"), row=2, col=1)
    fig_trends.add_trace(go.Scatter(x=trend_pd["period_str"], y=trend_pd["tpr_recall"]*100,
                                    mode="lines+markers", name="TPR/Recall (%)"), row=2, col=1)
    fig_trends.add_trace(go.Scatter(x=trend_pd["period_str"], y=trend_pd["f1"]*100,
                                    mode="lines+markers", name="F1 (%)"), row=2, col=1)
    fig_trends.add_trace(go.Scatter(x=trend_pd["period_str"], y=trend_pd["fn_rate"]*100,
                                    mode="lines+markers", name="Missed Fraud Rate (%)"), row=2, col=1)
    fig_trends.add_trace(go.Scatter(x=trend_pd["period_str"], y=trend_pd["mttd_days"],
                                    mode="lines+markers", name="MTTD (days)"), row=2, col=1)

    # Row 3: Value
    fig_trends.add_trace(go.Scatter(x=trend_pd["period_str"], y=trend_pd["total_fraud_value"],
                                    mode="lines+markers", name="Total Fraud Value"), row=3, col=1)
    fig_trends.add_trace(go.Scatter(x=trend_pd["period_str"], y=trend_pd["prevented_fraud_value"],
                                    mode="lines+markers", name="Prevented Fraud Value"), row=3, col=1)
    fig_trends.add_trace(go.Scatter(x=trend_pd["period_str"], y=trend_pd["avg_fraud_value"],
                                    mode="lines+markers", name="Avg Fraud Value"), row=3, col=1)

    fig_trends.update_layout(
        title_text=f"<b>{title} — Trends</b>",
        height=1000, width=1400,
        hovermode="x unified",
        showlegend=True,
        legend=dict(x=0.01, y=0.99)
    )

    fig_trends.show()


In [19]:
alerts = alerts_df
cases  = cases_df
tasks  = tasks_df

In [20]:
trend_sdf = build_fraud_trend_timeseries(
    alerts=alerts,
    cases=cases,
    period="LAST_90_DAYS",          # LAST_7_DAYS / LAST_30_DAYS / LAST_90_DAYS / LAST_YEAR / ALL_HISTORY
    granularity="weekly"            # daily / weekly / monthly / quarterly
)

trend_sdf = add_pop_indicators(trend_sdf)

render_fraud_trend_dashboard(trend_sdf)


26/04/17 06:50:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 06:50:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 06:50:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 06:50:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 06:50:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 06:50:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 0